# # US Accidents Severity — Prediction API


# 1: Load Prediction Engine

In [1]:
import pandas as pd
import joblib

# Load artifacts
model = joblib.load("Models/catboost_final_v4.pkl")
FEATURES = joblib.load("Models/features_v4.pkl")
threshold = joblib.load("Models/threshold_v4.pkl")

print(f"✅ Prediction Engine Ready. Optimized Threshold: {threshold:.4f}")

✅ Prediction Engine Ready. Optimized Threshold: 0.5034


# 2: Inference Function

In [2]:
def predict_severity(data):
    # Support both dict and DataFrame
    df_in = pd.DataFrame([data]) if isinstance(data, dict) else data.copy()
    
    # Define the EXACT list of categorical features used during training
    CAT_FEATURES = [
        'Day_of_Week', 'Month', 'Time_of_Day_Encoded', 'Timezone_Encoded',
        'Sunrise_Sunset_Encoded', 'Weather_Condition_Encoded', 'Wind_Direction_Encoded'
    ]
    
    # STRICTOR CONVERSION: Force all CAT_FEATURES to be strings
    # This prevents the "real number values" error (like Month: 1.0)
    for col in CAT_FEATURES:
        if col in df_in.columns:
            # Convert to float first to handle 1.0, then to int, then to string to get "1"
            # Or simply force to string. Let's do the safest version:
            df_in[col].astype(str).replace(r'\.0$', '', regex=True)
            
    # Calculate probability
    prob = model.predict_proba(df_in[FEATURES])[:, 1][0]
    risk = "High Risk (Severity 3-4)" if prob >= threshold else "Low Risk (Severity 1-2)"
    
    return {"Risk": risk, "Severity_Probability": f"{prob:.2%}"}

# 3: Real-Time Test

In [3]:
# Example Data Point
test_incident = {
    'Day_of_Week': 'Friday', 
    'Month': 1, 
    'Time_of_Day_Encoded': 2,
    'Timezone_Encoded': 1, 
    'Sunrise_Sunset_Encoded': 0, 
    'Weather_Condition_Encoded': 3,
    'Wind_Direction_Encoded': 1, 
    'Temperature_F': 35.0, 
    'Humidity_pct': 90.0,
    'Pressure_in': 29.8, 
    'Visibility_mi': 2.0, 
    'Wind_Speed_mph': 15.0,
    'Precipitation_in': 0.05, 
    'Hour': 17, 
    'Is_Weekend': 0, 
    'Is_Rush_Hour': 1,
    'Road_Features_Count': 2, 
    'Has_Traffic_Signal': 1, 
    'Has_Crossing': 0, 
    'Has_Junction': 1
}

# Fix: Use 'test_incident' to match the variable name above
result = predict_severity(test_incident)
print(f"Prediction Result: {result}")

Prediction Result: {'Risk': 'High Risk (Severity 3-4)', 'Severity_Probability': '71.55%'}


# 4: GENERATE BATCH PREDICTIONS CSV (Like Pipeline Style)

In [4]:
# ==========================================================
# GENERATE BATCH PREDICTIONS CSV (Complete & Fixed)
# ==========================================================
import pandas as pd
import numpy as np  # <--- Added this to fix your NameError
import joblib

# 1. Load data to predict on
df_all = pd.read_csv(r"F:\Depi Project\Final\Gold_Layer\US_Accidents_Gold_Modeling.csv")

# 2. Take a sample to act as a "test batch"
df_sample = df_all.sample(1000, random_state=42).copy()

# 3. Pre-process categorical columns
CAT_FEATURES = [
    'Day_of_Week', 'Month', 'Time_of_Day_Encoded', 'Timezone_Encoded',
    'Sunrise_Sunset_Encoded', 'Weather_Condition_Encoded', 'Wind_Direction_Encoded'
]
for col in CAT_FEATURES:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].astype(str).replace('\.0$', '', regex=True)

# 4. Generate Predictions
# Ensure FEATURES and threshold are loaded from your joblib files
probs = model.predict_proba(df_sample[FEATURES])[:, 1]
df_sample['Risk_Score_Prob'] = probs

# Use np.where (requires 'import numpy as np')
df_sample['Predicted_Risk_Level'] = np.where(probs >= threshold, 'High Risk (3-4)', 'Low Risk (1-2)')

# 5. Save the result for Power BI
prediction_output_path = r"F:\Depi Project\Final\Gold_Layer\Model_Predictions_Output.csv"
df_sample.to_csv(prediction_output_path, index=False)

print(f"✅ Success! File saved at: {prediction_output_path}")
print(df_sample[['Risk_Score_Prob', 'Predicted_Risk_Level']].head())

<>:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\dark-\AppData\Local\Temp\ipykernel_31932\2934298783.py:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  df_sample[col] = df_sample[col].astype(str).replace('\.0$', '', regex=True)


✅ Success! File saved at: F:\Depi Project\Final\Gold_Layer\Model_Predictions_Output.csv
        Risk_Score_Prob Predicted_Risk_Level
104241         0.442460       Low Risk (1-2)
199676         0.504262      High Risk (3-4)
140199         0.445134       Low Risk (1-2)
132814         0.713513      High Risk (3-4)
408697         0.430645       Low Risk (1-2)
